# 🏆 BIST KAZANAN STRATEJİ — Dürüst Backtest'in Canlı Motoru

Bu notebook, **DÜRÜST BACKTEST**'in bir devamıdır. O test 16 stratejiyi rastgele girişe karşı
sınadı; **yalnızca 5 tanesi** üç kapıyı birden geçti (rastgeleyi anlamlı geçti + t>2 + iki alt-dönemde
tutarlı). Bu notebook o 5 kazananı — ve **sadece onları** — canlı, para kazandırmaya yönelik tek bir
motora dönüştürür.

### Sınavı geçen 5 strateji (girdiğimiz tek gerçek edge kaynağı)
| Strateji | EXCESS_R | t | Rol |
|---|---|---|---|
| **DERIN_DEGER_BANKER** | **+0.261R** | 3.75 | En güçlü edge — nadir ama kral |
| ENGULFING | +0.066R | 9.16 | En yüksek istatistiksel güven (n büyük) |
| ICHIMOKU | +0.065R | 6.2 | Trend-teyitli kırılım |
| DERIN_DEGER | +0.039R | 4.62 | Kontrarian ucuzluk çekirdeği |
| TREND | +0.034R | 4.1 | Supertrend×ADX rejim filtresi |

### Bu motorun 3 ilkesi
1. **Sadece kanıtlanmış edge.** Elenen 11 strateji koda hiç girmez. Edge'i olmayanı taşımak = zarar taşımak.
2. **Konfluens = kalite.** Bir hisse aynı anda birden çok kazananı tetiklerse, sinyal daha güçlüdür.
   Adayları **edge-ağırlıklı konfluens skoruyla** sıralarız (banker sinyali tek başına bile en üstte).
3. **Aynı dürüst çıkış + risk yönetimi.** STOP=1ATR · +1.5R yarı-çıkış+başabaş · +3R · 10g.
   Pozisyon boyutu **sabit-kesir risk** (%1) ile belirlenir → tek işlem hesabı batıramaz.

> Önce 5 kazananı **taze veride yeniden doğrularız** (overfit değilse sayılar tekrar tutmalı), sonra
> ensemble'ı ölçer, en sonda **BUGÜNÜN canlı sinyallerini** giriş/stop/hedef/pozisyon ile üretiriz.
>
> **Veri (kilitli):** OHLCV → tvDatafeed · Evren → tradingview-screener `.limit(2000)` · yfinance YOK.
> Bu bir bildirim/karar-destek aracıdır; otomatik emir göndermez. Yatırım tavsiyesi değildir.

In [ ]:
# === HÜCRE 1: KURULUM ===
import subprocess,sys
def _pip(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p],check=False)
_pip("--upgrade","git+https://github.com/rongardF/tvdatafeed.git")
_pip("tradingview-screener","pandas","numpy","matplotlib","openpyxl")
import os,gc,warnings,numpy as np,pandas as pd,matplotlib.pyplot as plt
warnings.filterwarnings("ignore"); from datetime import datetime
plt.rcParams.update({"figure.facecolor":"#0d1117","axes.facecolor":"#0d1117","savefig.facecolor":"#0d1117",
  "text.color":"#e6edf3","axes.labelcolor":"#e6edf3","xtick.color":"#8b949e","ytick.color":"#8b949e",
  "axes.edgecolor":"#30363d","grid.color":"#21262d"})
try:
    from google.colab import drive; drive.mount("/content/drive"); BASE="/content/drive/MyDrive/BIST_Backtest"
except Exception: BASE="./BIST_Backtest"
os.makedirs(BASE,exist_ok=True); print("Klasör:",BASE)

In [ ]:
# === HÜCRE 2: AYARLAR (dürüst backtest ile birebir aynı çıkış + risk yönetimi) ===
CFG=dict(
  N_BARS=750,            # ~3 yıl günlük (yeniden-doğrulama 2 alt-döneme bölünür)
  MIN_FIYAT=1.0, MIN_LIKIT_MTL=10,
  ATR_LEN=14, ATR_STOP=1.0, T1_R=1.5, T2_R=3.0, SURE=10,
  MALIYET=0.0013,        # tek yön (round-trip ×2). %0.05 komisyon + %0.08 slipaj
  UNIVERSE_LIMIT=2000,
  # --- risk yönetimi (canlı sinyaller için) ---
  SERMAYE=100_000.0,     # örnek hesap büyüklüğü (TL) — kendi sermayenle değiştir
  RISK_PCT=0.01,         # işlem başına riske edilen sermaye oranı (%1)
  EDGE_TILT_MAX=2.0,     # yüksek-edge sinyallerde riski en fazla bu kat artır (banker için)
  MAX_ESZAMANLI=8,       # aynı anda önerilecek maksimum açık pozisyon (çeşitlendirme)
)
# Dürüst backtest'in ölçtüğü EXCESS_R (edge) — konfluens ağırlığı olarak kullanılır.
# Kaynak: sınavı geçen 5 stratejinin rapor edilen fazla-getirisi.
EDGE={"DERIN_DEGER_BANKER":0.261,"ENGULFING":0.066,"ICHIMOKU":0.065,"DERIN_DEGER":0.039,"TREND":0.034}
print("Çıkış herkese ortak: STOP=1ATR · +1.5R yarı+başabaş · +3R · 10g. Maliyet round-trip %{:.2f}"
      .format(CFG["MALIYET"]*2*100))
print("Risk: işlem başına %{:.0f} sermaye · edge-tilt ×{:.1f}'e kadar · maks {} eşzamanlı pozisyon"
      .format(CFG["RISK_PCT"]*100,CFG["EDGE_TILT_MAX"],CFG["MAX_ESZAMANLI"]))

In [ ]:
# === HÜCRE 3: VERİ (tvDatafeed + tradingview-screener) ===
from tvDatafeed import TvDatafeed,Interval
tv=TvDatafeed()   # temiz veri için TvDatafeed(kullanici,sifre)
def evren_getir():
    from tradingview_screener import Query
    _,df=(Query().set_markets("turkey").select("name","close","volume","average_volume_10d_calc")
          .limit(CFG["UNIVERSE_LIMIT"]).get_scanner_data())
    df=df.dropna(subset=["name"]).copy()
    df["likit_mTL"]=df["close"]*df["average_volume_10d_calc"]/1e6
    df=df[(df["close"]>=CFG["MIN_FIYAT"])&(df["likit_mTL"]>=CFG["MIN_LIKIT_MTL"])]
    s=sorted(df["name"].astype(str).unique()); print(f"Evren: {len(s)} hisse"); return s
def veri_indir(semboller):
    veri={}; hata=0
    for k,s in enumerate(semboller,1):
        if k%50==0: print(f"  {k}/{len(semboller)} indirildi ({len(veri)} ok)")
        try:
            df=tv.get_hist(s,exchange="BIST",interval=Interval.in_daily,n_bars=CFG["N_BARS"])
            if df is not None and len(df)>=150:
                veri[s]=df.rename(columns=str.lower)[["open","high","low","close","volume"]]
        except Exception: hata+=1
    print(f"İndirildi: {len(veri)} hisse ({hata} hata)"); return veri
# ÇALIŞTIR:
semboller=evren_getir(); veri=veri_indir(semboller)

In [ ]:
# === HÜCRE 4: GÖSTERGE KÜTÜPHANESİ (dürüst backtest ile BİREBİR — sinyal kayması olmasın) ===
def ema(s,n): return s.ewm(span=n,adjust=False).mean()
def atr(df,n=14):
    h,l,c=df["high"],df["low"],df["close"]; pc=c.shift()
    return pd.concat([h-l,(h-pc).abs(),(l-pc).abs()],axis=1).max(axis=1).ewm(alpha=1/n,adjust=False).mean()
def adx_di(df,n=14):
    h,l,c=df["high"],df["low"],df["close"]; pc=c.shift(); up=h.diff(); dn=-l.diff()
    plus=pd.Series(np.where((up>dn)&(up>0),up,0.0),index=df.index)
    minus=pd.Series(np.where((dn>up)&(dn>0),dn,0.0),index=df.index)
    tr=pd.concat([h-l,(h-pc).abs(),(l-pc).abs()],axis=1).max(axis=1).ewm(alpha=1/n,adjust=False).mean()
    pdi=100*plus.ewm(alpha=1/n,adjust=False).mean()/tr; mdi=100*minus.ewm(alpha=1/n,adjust=False).mean()/tr
    dx=100*(pdi-mdi).abs()/(pdi+mdi).replace(0,np.nan)
    return dx.ewm(alpha=1/n,adjust=False).mean(),pdi,mdi
def macd_h(s): m=ema(s,12)-ema(s,26); return m-ema(m,9)
def rsi(s,n=14):
    d=s.diff(); up=d.clip(lower=0).ewm(alpha=1/n,adjust=False).mean()
    dn=(-d.clip(upper=0)).ewm(alpha=1/n,adjust=False).mean()
    return 100-100/(1+up/dn.replace(0,np.nan))
def cmf(df,n=20):
    mfm=((df["close"]-df["low"])-(df["high"]-df["close"]))/(df["high"]-df["low"]).replace(0,np.nan)
    mfv=mfm*df["volume"]; return mfv.rolling(n).sum()/df["volume"].rolling(n).sum()
def bb_percent_b(s,n=20,k=2):
    m=s.rolling(n).mean(); sd=s.rolling(n).std(); return (s-(m-k*sd))/((2*k*sd).replace(0,np.nan))
def supertrend(df,n=10,mult=3):
    a=atr(df,n); hl2=(df["high"]+df["low"])/2; up=hl2-mult*a; dn=hl2+mult*a
    st=pd.Series(index=df.index,dtype=float); dirn=pd.Series(1,index=df.index)
    for i in range(1,len(df)):
        up.iloc[i]=max(up.iloc[i],up.iloc[i-1]) if df["close"].iloc[i-1]>up.iloc[i-1] else up.iloc[i]
        dn.iloc[i]=min(dn.iloc[i],dn.iloc[i-1]) if df["close"].iloc[i-1]<dn.iloc[i-1] else dn.iloc[i]
        dirn.iloc[i]=1 if df["close"].iloc[i]>dn.iloc[i-1] else (-1 if df["close"].iloc[i]<up.iloc[i-1] else dirn.iloc[i-1])
    return dirn
def williams_r(df,n=14):
    hh=df["high"].rolling(n).max(); ll=df["low"].rolling(n).min()
    return -100*(hh-df["close"])/(hh-ll).replace(0,np.nan)
def _ramp(x,lo,hi): return ((x-lo)/(hi-lo)).clip(0,1)*100
def boga_engulfing(df):
    po,pc=df["open"].shift(),df["close"].shift()
    return (pc<po)&(df["close"]>df["open"])&(df["close"]>=po)&(df["open"]<=pc)
def _edge(cond): return cond & (~cond.shift(1).fillna(False))   # yükselen kenar = ilk gün gir

In [ ]:
# === HÜCRE 5: SADECE 5 KAZANAN STRATEJİ (dürüst backtest'ten birebir) ===
def S_ICHIMOKU(df):
    h,l,c=df["high"],df["low"],df["close"]
    ten=(h.rolling(9).max()+l.rolling(9).min())/2; kij=(h.rolling(26).max()+l.rolling(26).min())/2
    a=((ten+kij)/2).shift(26); b=((h.rolling(52).max()+l.rolling(52).min())/2).shift(26)
    bulut=np.maximum(a,b); return _edge((c>bulut)&(ten>kij)&(c.shift(1)<=bulut.shift(1)))
def S_ENGULFING(df):
    c=df["close"]; return _edge(boga_engulfing(df)&(c>ema(c,20)))
def S_TREND(df):
    a,_,_=adx_di(df); return _edge((supertrend(df)>0)&(a>20))
# --- DERİN DEĞER KONTRARIAN (Deep_Value_Contrarian teknik çekirdeği) ---
def _tech_cheapness(df):
    c=df["close"]; r=rsi(c,14)
    hi52=df["high"].rolling(252,min_periods=60).max(); lo52=df["low"].rolling(252,min_periods=60).min()
    pos=(c-lo52)/(hi52-lo52).replace(0,np.nan); dd=(hi52-c)/hi52.replace(0,np.nan)
    pctb=bb_percent_b(c); ema200=c.ewm(span=200,adjust=False).mean(); dev=(c-ema200)/ema200
    wr=williams_r(df,14); rslow=rsi(c,70)
    comp=(0.28*_ramp(pos,0.85,0.05)+0.20*_ramp(r,50,15)+0.12*_ramp(rslow,55,25)
          +0.15*_ramp(dd,0.10,0.60)+0.10*_ramp(pctb,0.50,-0.05)+0.10*_ramp(-dev,0.0,0.25)
          +0.05*_ramp(-wr,20,85))
    return comp
def _banker(df):        # düşüşe rağmen para girişi (CMF pozitif + fiyat düşmüş)
    return (cmf(df,20)>0.02)&(df["close"].pct_change(20)<-0.02)
GATE=60.0
def S_DERIN_DEGER(df):        return _edge(_tech_cheapness(df)>=GATE)
def S_DERIN_DEGER_BANKER(df): return _edge((_tech_cheapness(df)>=GATE)&_banker(df))
# En güçlüden zayıfa (edge sırası). Konfluens ve raporlamada bu sıra kullanılır.
KAZANANLAR={"DERIN_DEGER_BANKER":S_DERIN_DEGER_BANKER,"ENGULFING":S_ENGULFING,
  "ICHIMOKU":S_ICHIMOKU,"DERIN_DEGER":S_DERIN_DEGER,"TREND":S_TREND}
print(f"{len(KAZANANLAR)} kazanan strateji yüklendi:",", ".join(KAZANANLAR))

In [ ]:
# === HÜCRE 6: BACKTEST MOTORU (dürüst backtest ile birebir) ===
def simule_islem(df,i,atr_i):
    n=len(df)
    if i+1>=n: return None
    giris=df["open"].iloc[i+1]; R=CFG["ATR_STOP"]*atr_i
    if R<=0 or not np.isfinite(giris): return None
    stop=giris-R; t1=giris+CFG["T1_R"]*R; t2=giris+CFG["T2_R"]*R
    mR=(giris*CFG["MALIYET"]*2)/R; yari=False; sc=stop
    for j in range(i+1,min(i+1+CFG["SURE"],n)):
        hi,lo=df["high"].iloc[j],df["low"].iloc[j]
        if not yari:
            if lo<=sc: return -1.0-mR
            if hi>=t2: return 3.0-mR
            if hi>=t1: yari=True; sc=giris
        else:
            if lo<=sc: return 0.75-mR
            if hi>=t2: return 2.25-mR
    kap=df["close"].iloc[min(i+CFG["SURE"],n-1)]; rr=(kap-giris)/R
    return (0.75+0.5*rr-mR) if yari else (rr-mR)
def backtest(veri,fn,rastgele=False,seed=0,tarih_filtre=None):
    rng=np.random.default_rng(seed); R=[]; T=[]
    for s,df in veri.items():
        if len(df)<120: continue
        if tarih_filtre: df=df[(df.index>=tarih_filtre[0])&(df.index<tarih_filtre[1])]
        if len(df)<120: continue
        a=atr(df,CFG["ATR_LEN"]).values; sig=fn(df).values.astype(bool)
        if rastgele:
            ns=int(sig[60:len(df)-CFG["SURE"]].sum())
            if ns==0: continue
            ad=np.arange(60,len(df)-CFG["SURE"]); sel=rng.choice(ad,min(ns,len(ad)),replace=False)
            sig=np.zeros(len(df),bool); sig[sel]=True
        i=60
        while i<len(df)-CFG["SURE"]:
            if sig[i] and np.isfinite(a[i]):
                r=simule_islem(df,i,a[i])
                if r is not None: R.append(r); T.append(df.index[i]); i+=CFG["SURE"]; continue
            i+=1
    return pd.DataFrame({"R":R,"tarih":T})
print("Motor hazır.")

In [ ]:
# === HÜCRE 7: 5 KAZANANI TAZE VERİDE YENİDEN DOĞRULA (overfit değilse sayılar tutmalı) ===
tarihler=sorted({t for df in veri.values() for t in df.index})
orta=tarihler[len(tarihler)//2]; d1=(tarihler[0],orta); d2=(orta,tarihler[-1])
dogrulama=[]
for ad,fn in KAZANANLAR.items():
    tam=backtest(veri,fn); rnd=backtest(veri,fn,rastgele=True,seed=7)
    h1=backtest(veri,fn,tarih_filtre=d1); h2=backtest(veri,fn,tarih_filtre=d2)
    if tam.empty: dogrulama.append({"strateji":ad,"n":0}); continue
    r=tam["R"]; rr=rnd["R"] if not rnd.empty else pd.Series([0.0])
    excess=r.mean()-rr.mean(); t_stat=r.mean()/(r.std()/np.sqrt(len(r))+1e-9)
    e1=h1["R"].mean() if not h1.empty else np.nan; e2=h2["R"].mean() if not h2.empty else np.nan
    tutarli=(not h1.empty and not h2.empty and e1>0 and e2>0)
    dogrulama.append({"strateji":ad,"n":len(r),"win%":round((r>0).mean()*100,1),
        "ort_R":round(r.mean(),3),"rastgele_R":round(rr.mean(),3),"EXCESS_R":round(excess,3),
        "PF":round(r[r>0].sum()/(-r[r<0].sum()+1e-9),2),"t_stat":round(t_stat,2),
        "H1_R":round(e1,3),"H2_R":round(e2,3),
        "HÂLÂ_GEÇER":"✓" if (excess>0.03 and t_stat>2 and tutarli) else "⚠"})
dg=pd.DataFrame(dogrulama).sort_values("EXCESS_R",ascending=False).reset_index(drop=True)
print(dg.to_string(index=False))
gecmeyen=dg[dg["HÂLÂ_GEÇER"]!="✓"]["strateji"].tolist()
if gecmeyen:
    print("\n⚠ Taze veride sınavı ARTIK geçemeyen(ler):",", ".join(gecmeyen),
          "\n  → Bu strateji(ler)in edge'i kırılgan olabilir; canlı sinyalde ağırlığı düşük tutulur.")
else:
    print("\n✅ 5 kazananın hepsi taze veride de sınavı geçti — edge tutarlı görünüyor.")

In [ ]:
# === HÜCRE 8: ENSEMBLE ÖLÇÜMÜ — "hepsi (birlik)" vs "konfluens ≥2" ===
# Her hisse için 5 kazananın sinyallerini üst üste bindirip iki ürünü ölçeriz:
#   (A) BIRLIK      : kazananlardan HERHANGİ biri tetiklerse gir (çok sinyal, geniş)
#   (B) KONFLUENS≥2 : en az 2 kazanan AYNI gün tetiklerse gir (az ama yüksek kalite)
def _konfluens_sayaci(df):
    m=np.zeros(len(df),int)
    for fn in KAZANANLAR.values(): m=m+fn(df).values.astype(int)
    return pd.Series(m,index=df.index)
def S_ENSEMBLE_BIRLIK(df):      return _konfluens_sayaci(df)>=1
def S_ENSEMBLE_KONFLUENS2(df):  return _konfluens_sayaci(df)>=2
urunler={"BIRLIK (≥1 kazanan)":S_ENSEMBLE_BIRLIK,"KONFLUENS (≥2 kazanan)":S_ENSEMBLE_KONFLUENS2}
ens=[]
for ad,fn in urunler.items():
    tam=backtest(veri,fn); rnd=backtest(veri,fn,rastgele=True,seed=7)
    if tam.empty: continue
    r=tam["R"]; rr=rnd["R"] if not rnd.empty else pd.Series([0.0])
    ens.append({"ürün":ad,"n":len(r),"win%":round((r>0).mean()*100,1),"ort_R":round(r.mean(),3),
        "rastgele_R":round(rr.mean(),3),"EXCESS_R":round(r.mean()-rr.mean(),3),
        "PF":round(r[r>0].sum()/(-r[r<0].sum()+1e-9),2),
        "t_stat":round(r.mean()/(r.std()/np.sqrt(len(r))+1e-9),2),
        "beklenti/işlem_R":round(r.mean(),3)})
et=pd.DataFrame(ens); print(et.to_string(index=False))
# Equity eğrileri (kümülatif R, maliyet dahil) — hangi ürün daha düzgün büyüyor?
fig,ax=plt.subplots(figsize=(11,5))
for ad,fn in urunler.items():
    tam=backtest(veri,fn)
    if tam.empty: continue
    eq=tam.sort_values("tarih")["R"].cumsum().reset_index(drop=True)
    ax.plot(eq.index,eq.values,label=f"{ad}  (Σ={eq.iloc[-1]:.0f}R, n={len(eq)})",lw=1.6)
ax.axhline(0,color="#f85149",lw=1,ls="--"); ax.legend(loc="upper left",facecolor="#161b22",edgecolor="#30363d")
ax.set_title("Ensemble kümülatif R (maliyet dahil) — konfluens daha az işlemde daha temiz getiri hedefler")
ax.set_xlabel("işlem sırası"); ax.set_ylabel("kümülatif R"); plt.tight_layout(); plt.show()
print("\nOKUMA: KONFLUENS≥2 genelde daha az işlem ama daha yüksek win%/PF verir → daha az komisyon,")
print("       daha az gürültü. BIRLIK daha çok fırsat yakalar. Canlı motor ikisini de skorlar.")

In [ ]:
# === HÜCRE 9: 🚀 CANLI SİNYAL MOTORU — BUGÜNÜN edge-ağırlıklı adayları + risk yönetimi ===
# Mantık: her hisse için 5 kazananın SON BAR'da tetikleyip tetiklemediğine bakarız.
# En az bir kazanan tetiklerse aday. Sıralama = edge-ağırlıklı KONFLUENS skoru:
#   skor = Σ EDGE[tetikleyen strateji]   (banker tek başına bile en üste çıkar)
# Giriş = bir sonraki seans açılışı (backtest kuralıyla aynı). Stop/hedef = ATR tabanlı.
# Pozisyon = sabit-kesir risk: lot = (SERMAYE*RISK_PCT*edge_tilt) / (ATR_STOP*ATR).
_edge_norm=np.median(list(EDGE.values()))
def _edge_tilt(skor):   # yüksek konfluens/edge → daha büyük (ama sınırlı) pozisyon
    return float(np.clip(skor/_edge_norm,1.0,CFG["EDGE_TILT_MAX"]))
def canli_sinyaller(veri):
    sat=[]
    # her strateji için son-bar sinyal serisini bir kez hesapla (hız)
    for s,df in veri.items():
        if len(df)<120: continue
        a=atr(df,CFG["ATR_LEN"])
        atr_son=a.iloc[-1]
        if not np.isfinite(atr_son) or atr_son<=0: continue
        tetik=[ad for ad,fn in KAZANANLAR.items() if bool(fn(df).iloc[-1])]
        if not tetik: continue
        skor=sum(EDGE[t] for t in tetik)
        kapanis=df["close"].iloc[-1]; giris=kapanis  # tahmini giriş ~ son kapanış (gerçek: yarın açılış)
        R=CFG["ATR_STOP"]*atr_son
        stop=giris-R; hedef1=giris+CFG["T1_R"]*R; hedef2=giris+CFG["T2_R"]*R
        tilt=_edge_tilt(skor)
        risk_tl=CFG["SERMAYE"]*CFG["RISK_PCT"]*tilt
        lot=int(risk_tl/R) if R>0 else 0
        maliyet_tl=lot*giris*CFG["MALIYET"]*2
        sat.append({"hisse":s,"konfluens":len(tetik),"edge_skor":round(skor,3),
            "stratejiler":"+".join(tetik),"son_bar":df.index[-1].date(),
            "kapanis":round(kapanis,2),"ATR":round(atr_son,2),
            "STOP":round(stop,2),"HEDEF_1.5R":round(hedef1,2),"HEDEF_3R":round(hedef2,2),
            "risk_tilt":round(tilt,2),"lot":lot,"risk_TL":round(lot*R,0),
            "pozisyon_TL":round(lot*giris,0),"tahmini_komisyon_TL":round(maliyet_tl,0)})
    df=pd.DataFrame(sat)
    if df.empty: return df
    return df.sort_values(["edge_skor","konfluens"],ascending=False).reset_index(drop=True)
sinyaller=canli_sinyaller(veri)
print("="*78)
if sinyaller.empty:
    print("Bugün 5 kazanandan hiçbiri son barda tetiklemedi. Sinyal yok → nakitte kal.")
else:
    n_goster=min(CFG["MAX_ESZAMANLI"],len(sinyaller))
    print(f"🚀 {len(sinyaller)} aday bulundu. Çeşitlendirme için en güçlü {n_goster} tanesi (edge-skor sırası):\n")
    goster=["hisse","stratejiler","konfluens","edge_skor","kapanis","STOP",
            "HEDEF_1.5R","HEDEF_3R","lot","risk_TL","pozisyon_TL"]
    print(sinyaller.head(n_goster)[goster].to_string(index=False))
    toplam_risk=sinyaller.head(n_goster)["risk_TL"].sum()
    print(f"\nEn güçlü {n_goster} pozisyonun TOPLAM riski: {toplam_risk:,.0f} TL "
          f"(sermayenin %{toplam_risk/CFG['SERMAYE']*100:.1f}'i).")
    if any(sinyaller.head(n_goster)["stratejiler"].str.contains("DERIN_DEGER_BANKER")):
        print("👑 Listede DERIN_DEGER_BANKER var — testteki en yüksek edge (+0.26R). Öncelikli aday.")
print("="*78)

In [ ]:
# === HÜCRE 10: SİNYALLERİ KAYDET (Excel) + ÖZET ===
stamp=datetime.now().strftime("%Y%m%d_%H%M")
if not sinyaller.empty:
    yol=os.path.join(BASE,f"canli_sinyaller_{stamp}.xlsx")
    with pd.ExcelWriter(yol) as xl:
        sinyaller.to_excel(xl,sheet_name="canli_sinyaller",index=False)
        dg.to_excel(xl,sheet_name="yeniden_dogrulama",index=False)
        et.to_excel(xl,sheet_name="ensemble",index=False)
    print("💾 Kaydedildi:",yol)
    print(f"\nGÜNÜN ÖZETİ ({stamp}):")
    print(f"  • Evren: {len(veri)} hisse tarandı")
    print(f"  • Aday sinyal: {len(sinyaller)}  ·  konfluens≥2 olan: {(sinyaller['konfluens']>=2).sum()}")
    ust=sinyaller.iloc[0]
    print(f"  • En güçlü aday: {ust['hisse']} ({ust['stratejiler']}) edge-skor={ust['edge_skor']}")
else:
    print("Kaydedilecek sinyal yok (bugün tetik yok).")

## 📖 Bu motoru nasıl kullanmalıyım

**Sıralama mantığı — neden edge-ağırlıklı konfluens?**
Dürüst backtest her stratejinin *fazla-getirisini* (EXCESS_R) ölçtü. Bir hisse birden çok kazananı aynı
anda tetikliyorsa, bağımsız edge'ler üst üste biner → daha yüksek beklenti. `edge_skor = Σ EDGE` bunu
tek sayıya indirir. DERIN_DEGER_BANKER'ın edge'i (+0.26R) diğerlerinin ~4 katı olduğu için, tek başına
bir banker sinyali bile listenin tepesine oturur — testin verdiği hüküm buydu.

**İşlem kuralları (backtest ile birebir — sapma = güvenilmez sonuç):**
- **Giriş:** sinyal son barda oluştuysa → *bir sonraki seans açılışında* al (tabloda `kapanis` yaklaşık
  giriştir; gerçek giriş yarınki açılış).
- **Stop:** `STOP` sütunu (giriş − 1×ATR). Buna sadık kal; edge istatistiği bu stopla ölçüldü.
- **Kısmi kâr:** +1.5R'de (`HEDEF_1.5R`) pozisyonun yarısını sat, kalanın stopunu başabaşa çek.
- **Tam kâr:** +3R'de (`HEDEF_3R`) kalanı kapat; hiçbiri olmazsa 10 işlem günü sonunda çık.

**Risk yönetimi (en kritik kısım — "en kazançlı" = en uzun hayatta kalan):**
- İşlem başına sermayenin **%1'i** riske edilir; `lot` bunu ATR'ye göre otomatik hesaplar.
- Yüksek-edge sinyallerde risk en fazla **×2** artar (`risk_tilt`) — banker sinyalleri biraz daha büyük.
- Aynı anda en çok **8 pozisyon**; toplam risk sermayenin ~%8-16'sını geçmemeli.
- `CFG["SERMAYE"]`yi kendi hesabınla değiştir; oranlar otomatik ölçeklenir.

**Dürüstlük notları (unutma):**
- Bu 5 strateji *canlı işlem defterinde* doğrulanana dek **yarı güvenle** kullanılmalı. Backtest edge'i
  gerçek edge'in *tahminidir*, garantisi değil.
- **Survivorship:** delist hisseler evrende yok → mutlak getiriler iyimser. Edge (rastgeleye karşı fark)
  bunu büyük ölçüde nötrler ama sıfırlamaz.
- **Konfluens az sinyal üretebilir.** Sinyal yoksa **nakitte kalmak da bir pozisyondur** — zorlama giriş yok.
- Bu araç karar-destektir; otomatik emir göndermez ve yatırım tavsiyesi değildir.